# Import and Install Dependencies

In [3]:
import os
import cv2
import numpy as np
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.utils import to_categorical
from keras.callbacks import TensorBoard
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

## draw style of landmarks like color, thickness and circle_radius

In [8]:
# Setup Folders for Collection
actions = np.array(['fine','i am']) # Actions that we try to detect
no_sequences = 500  # Thirty videos worth of data
sequence_length = 15  # Videos are going to be 30 frames in length

In [9]:
# Preprocess Data and Create Labels and Features
label_map = {label:num for num, label in enumerate(actions)} # create labels for actions like this {'hello': 0, 'thanks': 1, 'iloveyou': 2}

# array sequences: should be a big array containing all videos of all actions and this is features data or X data  
# array labels: should be a big array containing all labels of all actions and this is labels data or y data  
# Initialize an empty list to store the file paths
# Initialize an empty list to store the data

sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):

        res = np.load(os.path.join("/kaggle/input/iamfinez", action, "{}.npy".format(sequence)))

        # Reshape res from (30, 543, 3) to (30, 1629)
        res = res.reshape(res.shape[0], -1)    
        sequences.append(res)
        labels.append(label_map[action])

X = np.array(sequences) # sequences ---> (720, 30, 1629)
y = to_categorical(labels).astype(int) # labels --> (720, 2)

In [10]:
X.shape

(1000, 10, 1629)

In [11]:
# X_train --> (228, 30, 1629)
# y_train --> (228, 2)
# X_test --> (12, 30, 1629)
# y_test --> (12, 2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05) # 95% train and 5% test

In [12]:
actions.shape[0]

2

In [13]:
# Build and Train LSTM Neural Network
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(15,1629)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.fit(X_train, y_train, epochs=1000)
model.summary()

/opt/conda/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - categorical_accuracy: 0.6006 - loss: 0.6568
Epoch 2/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - categorical_accuracy: 0.7326 - loss: 0.5396
Epoch 3/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - categorical_accuracy: 0.8660 - loss: 0.3133
Epoch 4/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - categorical_accuracy: 0.8325 - loss: 0.4457
Epoch 5/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - categorical_accuracy: 0.9386 - loss: 0.1678
Epoch 6/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - categorical_accuracy: 0.9843 - loss: 0.0593
Epoch 7/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - categorical_accuracy: 0.7389 - loss: 2.4850
Epoch 8/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - categorical_accuracy: 0.9087 - loss: 0.2642
Epoch 9/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - categorical_accuracy: 0.9844 - loss: 0.0495
Epoch 10/1000
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - categorical_accuracy: 0.9844 - loss: 0.0572
Epoch 11/

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 15, 64)         │       433,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 15, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,764,584 (6.73 MB)

 Trainable params: 588,194 (2.24 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,176,390 (4.49 MB)

In [23]:
# Evaluation using Confusion Matrix and Accuracy
yhat = model.predict(X_test)

ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

print(accuracy_score(ytrue, yhat))

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
1.0


In [26]:
# save weights
model.save('iamfined.h5')